<a href="https://colab.research.google.com/github/EduardoAve/Labour-well-being/blob/main/data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Proyecto final de análisis exploratorio de datos
##Presentado por:
- Eduardo José Avendaño Caicedo
- Sebastián Dow Valenzuela



#Preparación de datos sobre Educadores en República Checa y Austria

##Introducción

Este informe documenta el proceso de limpieza y preparación de un conjunto de datos que recopila información sobre educadores en instituciones de educación superior en la República Checa y Austria. El dataset original consta de 129 columnas, cada una representando una pregunta o respuesta de una encuesta aplicada a los participantes. Sin embargo, en su estado inicial, los datos presentaban diversos problemas que dificultaban su análisis directo.

Nuestro objetivo fue transformar y estructurar el dataset de manera adecuada, asegurando la coherencia de sus variables y generando nuevas columnas derivadas de otras respuestas, lo que permitirá describir mejor ciertos aspectos clave de los educadores en el estudio. Para ello, se llevaron a cabo las siguientes acciones:

- Cálculo de nuevas variables mediante promedios y otras transformaciones, generando columnas objetivo que no estaban presentes en la versión original del dataset.
- Renombrado de columnas, asegurando nombres claros y uniformes.
- Corrección de tipos de datos, adaptando cada variable a su formato adecuado (numérico, categórico, etc.).
- Manejo de valores nulos e inconsistencias, para evitar sesgos y errores en el análisis.

Con estas modificaciones, el dataset ahora se encuentra estructurado y listo para ser utilizado en análisis posteriores.



In [337]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

In [338]:
df = pd.read_excel('/content/drive/MyDrive/labour well being/Data prepatartion/Dataset_Labour_Wellbeing.xlsx')

Se presenta una pequeña visualización del conjunto de datos inicial

In [339]:
df.head()

,Country,Version,1. Gender:,2. Age (in years):,Unnamed: 4,3. Nationality:,4. Current marital (partnership) status:,5. Do you currently care for underage children or dependent relatives?,6. The type of higher education insitution where you primarily work:,7. Subject area of the faculty (higher education institution) where you primarily work:,...,36. I tend to overwork.,37. I don’t consider my work to be finished until I am completely satisfied with the result.,38. My thoughts revolve around work almost exclusively.,"39. When I’m unsuccessful at work, it makes me feel very down.","40. If I don’t succeed at something, that just makes me all the more determined.",41. I can be calm and collected in almost all situations.,42. My life up till now has been characterized by professional success.,"43. By and large, I am happy and content.",44. I have the full support of my family.,What bothers you most about your work at a higher education institution at the moment? What do you like most?
0,2,4,2.0,28.0,2.0,Spanish,3.0,1.0,1.0,1.0,...,4.0,4.0,3.0,4.0,1.0,4.0,1.0,3.0,5.0,NaN
1,2,4,1.0,33.0,2.0,German,2.0,1.0,1.0,3.0,...,5.0,5.0,4.0,4.0,3.0,3.0,2.0,2.0,5.0,NaN
2,2,4,2.0,32.0,2.0,Kosovan,3.0,1.0,NaN,5.0,...,4.0,4.0,4.0,3.0,3.0,3.0,4.0,4.0,5.0,NaN
3,2,4,NaN,30.0,2.0,Spain,2.0,1.0,1.0,5.0,...,4.0,5.0,1.0,5.0,2.0,4.0,2.0,3.0,3.0,The low salary bothers me a lot. I like the mo...
4,2,4,1.0,39.0,2.0,Italian,2.0,2.0,1.0,5.0,...,5.0,5.0,3.0,3.0,3.0,2.0,4.0,4.0,5.0,The competition model


In [340]:
# Renombrar la columna desconocida
df = df.rename(columns={'Unnamed: 4': 'Unknown_Column'})

#Creación de columnas de interés para el análisis


In [341]:
# --- 1. Función de Inversión ---
def invert_scale(series):
    """Invierte valores en una escala de 1 a 5, ignorando NaNs."""
    return series.apply(lambda x: 6 - x if pd.notna(x) else x)

# --- 2. Cálculo de Variables (Bloque 1: Academic Resources, etc.) ---
print("Calculando variables: Academic Resources, Autonomy, Leadership, Community, Satisfaction, Burnout...")
df['Academic_resources'] = df.iloc[:, 36:42].mean(axis=1) # Cols 37-42
df['Performance_pressure'] = df.iloc[:, 42] # Col 43
df['Perceived_autonomy'] = ( # Cols 44-49 (Indices 43-48)
    df.iloc[:, 43] +
    invert_scale(df.iloc[:, 44]) +
    invert_scale(df.iloc[:, 45]) +
    df.iloc[:, 46] +
    df.iloc[:, 47] +
    invert_scale(df.iloc[:, 48])
) / 6
df['Quality_of_leadership'] = df.iloc[:, 49:53].mean(axis=1) # Cols 50-53
df['Sense_of_community'] = df.iloc[:, 53:56].mean(axis=1) # Cols 54-56
df['Job_satisfaction'] = df.iloc[:, 56:61].mean(axis=1) # Cols 57-61
df['Burnout'] = df.iloc[:, 80:84].mean(axis=1) # Cols 81-84
print("Cálculo Bloque 1 completado.")

# --- 3. Cálculo de Variables de Motivación (Bloque 2 - CON ÍNDICES CORREGIDOS) ---
print("Calculando variables de Motivación Laboral (con índices corregidos)...")
# Índices 0-based correspondientes a las columnas 62-80 que especificaste
amotivation_indices = [61, 67, 73]         # Cols: 62, 68, 74
extrinsic_social_indices = [62, 68, 74]    # Cols: 63, 69, 75
extrinsic_material_indices = [63, 69, 75]  # Cols: 64, 70, 76
introjected_indices = [64, 70, 76, 79]     # Cols: 65, 71, 77, 80
identified_indices = [65, 71, 77]          # Cols: 66, 72, 78
intrinsic_indices = [66, 72, 78]           # Cols: 67, 73, 79

df['Amotivation'] = df.iloc[:, amotivation_indices].mean(axis=1)
df['Extrinsic_Social'] = df.iloc[:, extrinsic_social_indices].mean(axis=1)
df['Extrinsic_Material'] = df.iloc[:, extrinsic_material_indices].mean(axis=1)
df['Introjected'] = df.iloc[:, introjected_indices].mean(axis=1)
df['Identified'] = df.iloc[:, identified_indices].mean(axis=1)
df['Intrinsic'] = df.iloc[:, intrinsic_indices].mean(axis=1)
print("Cálculo Bloque 2 (Motivación) completado.")

# --- 4. Definición de Columnas Originales a Eliminar (Como en tu código base) ---
# Esta lista ya incluye correctamente los rangos fuente de TODAS las variables calculadas
# y las otras secciones a eliminar.
print("Definiendo columnas originales a eliminar...")
cols_to_drop = (
    list(range(36, 42)) +  # Academic resources originales (Cols 37-42)
    [42] +                 # Performance pressure original (Col 43)
    list(range(43, 49)) +  # Perceived autonomy originales (Cols 44-49)
    list(range(49, 53)) +  # Quality of leadership originales (Cols 50-53)
    list(range(53, 56)) +  # Sense of community originales (Cols 54-56)
    list(range(56, 61)) +  # Job satisfaction originales (Cols 57-61)
    list(range(80, 84)) +  # Burnout originales (Cols 81-84)
    # Fuentes de Motivación (Índices 61-79 -> Cols 62-80) - YA INCLUIDO ABAJO
    list(range(61, 80)) +  # Work motivation ORIGINALES (Índices 61-79 -> Cols 62-80)
    list(range(84, 106)) + # Vulnerability to burnout (Índices 84-105 -> Cols 85-106)
    list(range(106, 129))  # Columnas adicionales (Índices 106-128 -> Cols 107-129)
)
# Asegurarse de que no haya duplicados (aunque range() y listas no deberían crearlos aquí)
# y ordenar puede ser útil para la legibilidad, pero no estrictamente necesario para df.columns[]
cols_to_drop_indices = sorted(list(set(cols_to_drop)))
print(f"Se definieron {len(cols_to_drop_indices)} índices únicos para eliminar.")

# --- 5. Eliminación de Columnas Originales ---
print("Eliminando columnas originales...")
# Usar df.columns[indices] para obtener nombres y eliminar por nombre es más seguro
# si la estructura de df pudiera cambiar inesperadamente, pero drop por índice también funciona.
# Mantenemos tu enfoque original de drop por índice con inplace=True.
try:
    # Obtener los nombres de las columnas correspondientes a los índices ANTES de eliminar
    # Esto puede fallar si algún índice está fuera de rango después de añadir columnas,
    # por eso el enfoque de "mantener" es a veces más seguro, pero probemos tu método.
    columns_to_drop_names = df.columns[cols_to_drop_indices]
    df.drop(columns=columns_to_drop_names, inplace=True)
    # Alternativa directa por índice (menos recomendada si se añadieron muchas cols):
    # df.drop(df.columns[cols_to_drop_indices], axis=1, inplace=True)
    print("Columnas originales eliminadas con éxito.")
except IndexError as e:
    print(f"Error al intentar eliminar columnas por índice: {e}")
    print("Esto puede ocurrir si los índices ya no son válidos después de añadir columnas.")
    print("Considera usar el enfoque de 'mantener columnas' del bloque anterior si este falla.")


# --- Verificación Final ---
print("\nDataFrame después de calcular todas las variables y eliminar originales:")
# Verifica si el drop funcionó (si no hubo error)
if 'Academic_resources' in df.columns and 36 in cols_to_drop_indices:
     # Si la columna calculada existe Y su índice fuente estaba en la lista de drop,
     # es probable que el drop por índice no funcionara como se esperaba si las columnas se añadieron al final.
     # Sin embargo, si el drop por nombre funcionó, las columnas originales no deberían estar.
     pass # La verificación real es ver las columnas restantes

print(f"Número de columnas final: {len(df.columns)}")
print("Primeras filas y columnas:")
print(df.head())
# print(df.info()) # Descomenta para ver info completa

Calculando variables: Academic Resources, Autonomy, Leadership, Community, Satisfaction, Burnout...
Cálculo Bloque 1 completado.
Calculando variables de Motivación Laboral (con índices corregidos)...
Cálculo Bloque 2 (Motivación) completado.
Definiendo columnas originales a eliminar...
Se definieron 93 índices únicos para eliminar.
Eliminando columnas originales...
Columnas originales eliminadas con éxito.

DataFrame después de calcular todas las variables y eliminar originales:
Número de columnas final: 49
Primeras filas y columnas:
   Country  Version  1. Gender:  2. Age (in years):  Unknown_Column  \
0        2        4         2.0                28.0             2.0   
1        2        4         1.0                33.0             2.0   
2        2        4         2.0                32.0             2.0   
3        2        4         NaN                30.0             2.0   
4        2        4         1.0                39.0             2.0   

  3. Nationality:  4. Current mar

In [342]:
#Visualizamos como quedaron nuestras nuevas columnas calculadas
df.head()

,Country,Version,1. Gender:,2. Age (in years):,Unknown_Column,3. Nationality:,4. Current marital (partnership) status:,5. Do you currently care for underage children or dependent relatives?,6. The type of higher education insitution where you primarily work:,7. Subject area of the faculty (higher education institution) where you primarily work:,...,Quality_of_leadership,Sense_of_community,Job_satisfaction,Burnout,Amotivation,Extrinsic_Social,Extrinsic_Material,Introjected,Identified,Intrinsic
0,2,4,2.0,28.0,2.0,Spanish,3.0,1.0,1.0,1.0,...,3.00,2.666667,2.2,4.00,1.333333,2.000000,1.333333,3.25,6.333333,6.333333
1,2,4,1.0,33.0,2.0,German,2.0,1.0,1.0,3.0,...,2.00,1.666667,2.4,4.00,2.666667,3.000000,3.666667,5.50,6.333333,5.000000
2,2,4,2.0,32.0,2.0,Kosovan,3.0,1.0,NaN,5.0,...,5.00,5.000000,5.0,4.25,2.000000,4.000000,5.000000,5.50,6.000000,6.000000
3,2,4,NaN,30.0,2.0,Spain,2.0,1.0,1.0,5.0,...,2.00,3.000000,2.8,3.25,2.666667,3.333333,2.333333,5.25,4.000000,4.333333
4,2,4,1.0,39.0,2.0,Italian,2.0,2.0,1.0,5.0,...,3.75,3.000000,2.6,4.50,1.000000,1.000000,3.666667,6.00,6.333333,5.333333


#La columna 25 es un duplicado de la 26, se eliminará.


In [343]:
# Eliminar la primera columna "14.1" (con el índice 24)
df = df.drop(df.columns[24], axis=1)

#A continuación, ponemos nombres más representativos para las columnas y que permitan un trabajo más cómodo:

In [344]:
# Diccionario de renombrado
rename_dict = {
    "Country": "Country",
    "Version": "Version",
    "1. Gender:": "Gender",
    "2. Age (in years):": "Age",
    "Unknown_Column": "Unknown_Column",
    "3. Nationality:": "Nationality",
    "4. Current marital (partnership) status:": "Marital_Status",
    "5. Do you currently care for underage children or dependent relatives?": "Care_Responsibilities",
    "6. The type of higher education insitution where you primarily work:": "HEI_Type",
    "7. Subject area of the faculty (higher education institution) where you primarily work:": "Faculty_Subject_Area",
    "8. Duration of your current employment contract at the higher education institution where you primarily work:": "Employment_Contract_Duration",
    "9. Extent of employment in higher education (in hours/week, aggregated for all higher education institutions where you work):": "HEI_Employment_Hours",
    "10. Actual average weekly working hours in higher education (in a typical semester week):": "HEI_Actual_Weekly_Hours",
    "Effort (less, more, equal)": "Effort_Level",
    "Effort [%]": "Effort_Percentage",
    "Income CZK": "Income_CZK",
    "Income EUR": "Income_EUR",
    "Income EURO": "Income_EURO",
    "Euro Adj.": "Euro_Adjusted",
    "Salary/hour": "Salary_per_Hour",
    "Salary effort/hour": "Salary_Effort_per_Hour",
    "12. Do you hold a leadership position at a higher education institution?": "Leadership_Position",
    "13. How influential are you in helping to shape key academic policies at your institution at the level of department or similar unit?": "Policy_Influence",
    "14. Do you currently have another (paid) job outside higher education?": "Other_Paid_Job",
    "14.1. Actual average weekly working hours outside higher education (in a typical semester week):": "Other_Job_Weekly_Hours",
    "14.1. Actual average weekly working hours outside higher education (in a typical semester week): .1": "Other_Job_Weekly_Hours_1",
    "Academic/Non-academic": "Academic_or_Non_Academic",
    "CZ_15. Your current position at the higher education institution, where you primarily work: ": "Current_Position_CZ",
    "AT_15. Your current position at the higher education institution, where you primarily work: ": "Current_Position_AT",
    "16. Please choose the category that best fits your job description:": "Job_Category",
    "17. The highest level of education attained:": "Highest_Education_Level",
    "18. Total length of your career in Czech higher education in years:": "Career_Length_CZ",
    "1. Teaching (classroom instruction, preparation of instructional materials and lesson plans, advising students, reading and evaluating student work, examination management, etc.)": "Teaching_Hours",
    "2. Research (reading literature, designing and conducting experiments, collecting and analysing data, writing articles or other scientific texts, etc.)": "Research_Hours",
    "3. Activities related to externally funded research projects (searching for information on available funding sources, preparation of grant applications and project reports, project management and administration, etc.)": "Funded_Research_Activities",
    "4. Organisational and administrative activities (organising and attending meetings, dealing with tasks and documents not directly related to teaching, research, or externally funded research projects, etc.)": "Administrative_Activities",
    "Performance_pressure": "Performance_Pressure",
    "Perceived_autonomy": "Perceived_Autonomy",
    "Quality_of_leadership": "Quality_of_Leadership",
    "Sense_of_community": "Sense_of_Community",
    "Job_satisfaction": "Job_Satisfaction",
    "Burnout": "Burnout"
}

# Renombrar columnas
df = df.rename(columns=rename_dict)

#Los nuevos nombres de las columnas quedaron de la siguiente manera:

In [345]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2748 entries, 0 to 2747
Data columns (total 48 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Country                       2748 non-null   int64  
 1   Version                       2748 non-null   int64  
 2   Gender                        2739 non-null   float64
 3   Age                           2699 non-null   float64
 4   Unknown_Column                2707 non-null   float64
 5   Nationality                   270 non-null    object 
 6   Marital_Status                2720 non-null   float64
 7   Care_Responsibilities         2729 non-null   float64
 8   HEI_Type                      2744 non-null   float64
 9   Faculty_Subject_Area          2720 non-null   float64
 10  Employment_Contract_Duration  2745 non-null   float64
 11  HEI_Employment_Hours          2712 non-null   object 
 12  HEI_Actual_Weekly_Hours       2558 non-null   float64
 13  Eff

Las columnas de "HEI_Employment_Hours", "Income_EUR", "Career_Length_CZ", y "Administrative_Activities" no deberían ser objetos, hacemos el cambio de tipo de dato.  

In [346]:
df["HEI_Employment_Hours"] = pd.to_numeric(df["HEI_Employment_Hours"], errors="coerce")
df["Income_EUR"] = pd.to_numeric(df["Income_EUR"], errors="coerce")
df["Career_Length_CZ"] = pd.to_numeric(df["Career_Length_CZ"], errors="coerce")
df["Administrative_Activities"] = pd.to_numeric(df["Administrative_Activities"], errors="coerce")


La varaible de Income EUR0 contiene todos los registros que necesitamos sobre el dinero que ganan los trabajadores, por lo tanto, podemos prescindir de Income_CZK e Income_EUR

In [347]:
df.drop(columns=['Income_CZK', 'Income_EUR'], inplace=True)

Las variables 'Current_Position_CZ' y 'Current_Position_AT' representan la misma información sobre la posición actual de un individuo, pero diferenciadas por ubicación geográfica. Para evitar redundancia y facilitar el análisis, se consolidarán en una única variable, 'Current_Position'.

In [348]:
# Crear la nueva columna 'Current_Position' combinando 'Current_Position_CZ' y 'Current_Position_AT'
df['Current_Position'] = df[['Current_Position_CZ', 'Current_Position_AT']].sum(axis=1, skipna=True)

# Eliminar las columnas originales
df.drop(columns=['Current_Position_CZ', 'Current_Position_AT'], inplace=True)

#Tratamiento de valores nulos

Eliminamos la columna desconocida ya que no es útil para el análisis de los datos

In [349]:
df.drop(columns=['Unknown_Column'], inplace=True)

La variable de "Nationality" parece ser de gran interés, sin embargo, tiene muy pocos valores por lo que será mejor prescindir de ella.

In [350]:
df.drop(columns=['Nationality'], inplace=True)

Las siguientes variables de interés contienen valores nulos que consideramos que pueden ser inputados utilizando la mediana para no perder tanta información en los datos

La variable Other_Paid_Job puede llenarse con 1, lo que representa que el trabajador no tiene otro trabajo pagado.

In [351]:
df["Other_Paid_Job"] = df["Other_Paid_Job"].fillna(1)

Si bien algunas variables presentan un alto porcentaje de valores nulos, esto se debe a la estructura del conjunto de datos y no necesariamente a errores en la recopilación. En este caso, el conjunto de datos distingue entre personal académico y no académico dentro de las instituciones educativas, lo que explica la ausencia de datos en ciertos campos dependiendo del tipo de empleado. Específicamente, las variables 'Teaching_Hours', 'Research_Hours', 'Funded_Research_Activities' y 'Administrative_Activities' corresponden exclusivamente al personal académico, mientras que 'Job_Category', 'Highest_Education_Level' y 'Career_Length_CZ' están más relacionadas con roles no académicos. Por esta razón, la presencia de valores nulos en estas variables es esperada y no afecta la calidad del conjunto de datos.

Los valores nulos en las variables "Teaching_Hours", "Research_Hours", "Funded_Research_Activities" y "Administrative_Activities" dentro del grupo de académicos pueden interpretarse como ceros, representando la ausencia de horas dedicadas a dichas actividades específicas.

#Como resultado del proceso de limpieza e imputación de datos, obtenemos el conjunto de datos final listo para su análisis.


In [352]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2748 entries, 0 to 2747
Data columns (total 43 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Country                       2748 non-null   int64  
 1   Version                       2748 non-null   int64  
 2   Gender                        2739 non-null   float64
 3   Age                           2699 non-null   float64
 4   Marital_Status                2720 non-null   float64
 5   Care_Responsibilities         2729 non-null   float64
 6   HEI_Type                      2744 non-null   float64
 7   Faculty_Subject_Area          2720 non-null   float64
 8   Employment_Contract_Duration  2745 non-null   float64
 9   HEI_Employment_Hours          2711 non-null   float64
 10  HEI_Actual_Weekly_Hours       2558 non-null   float64
 11  Effort_Level                  2669 non-null   float64
 12  Effort_Percentage             2553 non-null   float64
 13  Inc

In [353]:
# Define las columnas donde los NaN son estructurales/esperados
structural_nan_cols = [
    'Job_Category', 'Highest_Education_Level', 'Career_Length_CZ',
    'Teaching_Hours', 'Research_Hours', 'Funded_Research_Activities',
    'Administrative_Activities'
]

print(f"DataFrame inicial: {df.shape[0]} filas, {df.shape[1]} columnas")

# ==============================================================================
# PASO 1: Filtrar Filas por Límite de NaNs (Umbral=5, excluyendo estructurales)
# ==============================================================================
print("\n--- PASO 1: Filtrando filas con > 5 NaNs (no estructurales) ---")

max_allowed_nans = 5
all_cols_step1 = df.columns.tolist()
cols_to_check_nans_step1 = [col for col in all_cols_step1 if col not in structural_nan_cols]
nan_counts_per_row_step1 = df[cols_to_check_nans_step1].isnull().sum(axis=1)
rows_to_keep_mask_step1 = nan_counts_per_row_step1 <= max_allowed_nans

rows_before_step1 = len(df)
df = df[rows_to_keep_mask_step1].copy() # Aplicar filtro y sobreescribir df
rows_after_step1 = len(df)

print(f"Filas antes: {rows_before_step1}")
print(f"Filas después: {rows_after_step1}")
print(f"Filas eliminadas en Paso 1: {rows_before_step1 - rows_after_step1}")

# ==============================================================================
# PASO 2: Eliminar Outliers de Ingresos con IQR
# ==============================================================================
print("\n--- PASO 2: Eliminando outliers de Ingresos (IQR) ---")

cols_for_iqr = ['Income_EURO', 'Euro_Adjusted']
rows_to_keep_mask_step2 = pd.Series(True, index=df.index) # Empezar manteniendo todo

for col in cols_for_iqr:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f"  Límites IQR para {col}: Inferior={lower_bound:.2f}, Superior={upper_bound:.2f}")

    outliers_mask_col = (df[col] < lower_bound) | (df[col] > upper_bound)
    print(f"  Outliers detectados en {col}: {outliers_mask_col.sum()}")
    rows_to_keep_mask_step2 = rows_to_keep_mask_step2 & (~outliers_mask_col)

rows_before_step2 = len(df)
df = df[rows_to_keep_mask_step2].copy() # Aplicar filtro y sobreescribir df
rows_after_step2 = len(df)

print(f"Filas antes: {rows_before_step2}")
print(f"Filas después: {rows_after_step2}")
print(f"Filas eliminadas en Paso 2: {rows_before_step2 - rows_after_step2}")

# ==============================================================================
# PASO 3: Imputación de Valores Faltantes con KNN
# ==============================================================================
print("\n--- PASO 3: Imputando NaNs restantes con KNN (k=5) ---")

# Columnas donde queremos imputar los NaNs (numéricas con NaNs restantes)
cols_to_impute_step3 = [
    'Age', 'HEI_Actual_Weekly_Hours', 'Effort_Level', 'Effort_Percentage',
    'Income_EURO', 'Euro_Adjusted', 'Salary_per_Hour', 'Salary_Effort_per_Hour',
    'Academic_resources', 'Perceived_Autonomy',
    # Añadir otras columnas NUMÉRICAS si aún tienen NaNs y quieres imputarlas
    # Ejemplo: 'HEI_Employment_Hours' si tuviera NaNs
]
# Verificar que estas columnas existen en df
cols_to_impute_step3 = [col for col in cols_to_impute_step3 if col in df.columns]

# Identificar columnas numéricas para usar en KNN (excluyendo estructurales y categóricas/IDs)
numeric_cols_step3 = df.select_dtypes(include=np.number).columns.tolist()
cols_to_exclude_from_matrix_step3 = list(set(structural_nan_cols + [
    'Country', 'Version', 'Gender', 'Marital_Status', 'Care_Responsibilities',
    'HEI_Type', 'Faculty_Subject_Area', 'Employment_Contract_Duration',
    'Leadership_Position', 'Policy_Influence', 'Other_Paid_Job',
    'Other_Job_Weekly_Hours_1', 'Academic_or_Non_Academic', 'Current_Position',
    # También excluir las estructurales por si acaso
    'Job_Category', 'Highest_Education_Level', 'Career_Length_CZ',
    'Teaching_Hours', 'Research_Hours', 'Funded_Research_Activities',
    'Administrative_Activities'
]))

numeric_cols_for_imputation_step3 = [
    col for col in numeric_cols_step3
    if col not in cols_to_exclude_from_matrix_step3
]

print(f"Se usarán {len(numeric_cols_for_imputation_step3)} columnas numéricas para KNN.")

if not numeric_cols_for_imputation_step3:
    print("¡Advertencia! No se encontraron columnas numéricas adecuadas para KNN.")
else:
    # Seleccionar el subconjunto numérico
    df_numeric_subset = df[numeric_cols_for_imputation_step3]
    numeric_subset_cols = df_numeric_subset.columns
    numeric_subset_index = df_numeric_subset.index

    # Verificar si hay NaNs que imputar en este subset
    if df_numeric_subset.isnull().sum().sum() > 0:
        # Escalar
        print("  Escalando datos...")
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_numeric_subset)

        # Imputar
        print("  Aplicando KNNImputer...")
        imputer = KNNImputer(n_neighbors=5)
        imputed_scaled_data = imputer.fit_transform(scaled_data)

        # Invertir Escalado
        print("  Invirtiendo escalado...")
        imputed_original_scale_data = scaler.inverse_transform(imputed_scaled_data)

        # Crear DataFrame temporal con datos imputados
        df_imputed_subset = pd.DataFrame(imputed_original_scale_data, columns=numeric_subset_cols, index=numeric_subset_index)

        # Actualizar DataFrame principal
        print("  Actualizando DataFrame principal...")
        for col in numeric_cols_for_imputation_step3:
            if col in df_imputed_subset.columns:
                 df[col] = df_imputed_subset[col]

        print("  Verificando NaNs en columnas imputadas (deberían ser 0):")
        print(df[cols_to_impute_step3].isnull().sum())
    else:
        print("  No se encontraron NaNs en las columnas seleccionadas para KNN, se omite la imputación.")

# ==============================================================================
# PASO 4: Eliminar Filas con NaNs Restantes (Excluyendo Estructurales)
# ==============================================================================
print("\n--- PASO 4: Eliminando filas con NaNs restantes (excl. estructurales) ---")

all_cols_step4 = df.columns.tolist()
cols_to_check_for_nans_step4 = [col for col in all_cols_step4 if col not in structural_nan_cols]

print(f"Se revisarán {len(cols_to_check_for_nans_step4)} columnas para eliminar filas con NaNs.")

rows_before_step4 = len(df)
# Aplicar dropna sobre el subconjunto de columnas no estructurales
df_final_cleaned = df.dropna(subset=cols_to_check_for_nans) # No necesita .copy() aquí
rows_after_step4 = len(df_final_cleaned)

print(f"Filas antes: {rows_before_step4}")
print(f"Filas después: {rows_after_step4}")
print(f"Filas eliminadas en Paso 4: {rows_before_step4 - rows_after_step4}")

# --- Verificación Final del Paso 4 ---
nan_check_sum_step4 = df_final_cleaned[cols_to_check_for_nans].isnull().sum().sum()
print(f"\nSuma total de NaNs en columnas no estructurales después del dropna final: {nan_check_sum_step4}")

if nan_check_sum_step4 != 0:
    print("¡¡¡ADVERTENCIA!!! Todavía quedan NaNs inesperados en columnas no estructurales.")
    print("Revisa las listas de columnas y los pasos anteriores.")
    print(df_final_cleaned.isnull().sum())
else:
    print("Verificación completada: No quedan NaNs en columnas no estructurales.")
    print("\nNaNs restantes por columna (solo deberían quedar en estructurales):")
    print(df_final_cleaned.isnull().sum())


# --- Resultado Final ---
print("\n==============================================================================")
print("PROCESO DE LIMPIEZA COMPLETADO")
print(f"El DataFrame final 'df_final_cleaned' tiene: {df_final_cleaned.shape[0]} filas y {df_final_cleaned.shape[1]} columnas.")
print("==============================================================================")

DataFrame inicial: 2748 filas, 43 columnas

--- PASO 1: Filtrando filas con > 5 NaNs (no estructurales) ---
Filas antes: 2748
Filas después: 2671
Filas eliminadas en Paso 1: 77

--- PASO 2: Eliminando outliers de Ingresos (IQR) ---
  Límites IQR para Income_EURO: Inferior=-1904.77, Superior=7062.86
  Outliers detectados en Income_EURO: 146
  Límites IQR para Euro_Adjusted: Inferior=-1214.88, Superior=6268.01
  Outliers detectados en Euro_Adjusted: 151
Filas antes: 2671
Filas después: 2520
Filas eliminadas en Paso 2: 151

--- PASO 3: Imputando NaNs restantes con KNN (k=5) ---
Se usarán 22 columnas numéricas para KNN.
  Escalando datos...
  Aplicando KNNImputer...
  Invirtiendo escalado...
  Actualizando DataFrame principal...
  Verificando NaNs en columnas imputadas (deberían ser 0):
Age                        0
HEI_Actual_Weekly_Hours    0
Effort_Level               0
Effort_Percentage          0
Income_EURO                0
Euro_Adjusted              0
Salary_per_Hour            0
Sal

#Limpieza de valores academic vs non-academic

In [354]:
df = df_final_cleaned.copy() # Trabajar sobre una copia

print("--- PASO 5: Eliminando filas con NaNs restantes DENTRO de subgrupos ---")
print(f"Total de filas ANTES de la eliminación por subgrupo: {len(df)}")

# --- Definir columnas estructurales por grupo ---
cols_academic_structural = [
    'Teaching_Hours', 'Research_Hours', 'Funded_Research_Activities', 'Administrative_Activities'
]
cols_non_academic_structural = [
    'Job_Category', 'Highest_Education_Level', 'Career_Length_CZ'
]

# --- Filtrar y Limpiar Subgrupo Académico ---
# Usando la codificación correcta: 2 = Académico
df_academic = df[df['Academic_or_Non_Academic'] == 2].copy() # Usar .copy() para evitar warnings
rows_before_academic = len(df_academic)
print(f"\n--- Subgrupo: Académicos (Academic_or_Non_Academic == 2) ---")
print(f"Filas antes de limpiar: {rows_before_academic}")
print("NaNs a eliminar en columnas relevantes para Académicos:")
print(df_academic[cols_academic_structural].isnull().sum())
# Eliminar filas con NaNs en las columnas estructurales académicas
df_academic.dropna(subset=cols_academic_structural, inplace=True)
rows_after_academic = len(df_academic)
print(f"Filas después de limpiar: {rows_after_academic}")
print(f"Filas eliminadas en subgrupo Académico: {rows_before_academic - rows_after_academic}")


# --- Filtrar y Limpiar Subgrupo No Académico ---
# Usando la codificación correcta: 1 = No Académico
df_non_academic = df[df['Academic_or_Non_Academic'] == 1].copy() # Usar .copy()
rows_before_non_academic = len(df_non_academic)
print(f"\n--- Subgrupo: No Académicos (Academic_or_Non_Academic == 1) ---")
print(f"Filas antes de limpiar: {rows_before_non_academic}")
print("NaNs a eliminar en columnas relevantes para No Académicos:")
print(df_non_academic[cols_non_academic_structural].isnull().sum())
# Eliminar filas con NaNs en las columnas estructurales no académicas
df_non_academic.dropna(subset=cols_non_academic_structural, inplace=True)
rows_after_non_academic = len(df_non_academic)
print(f"Filas después de limpiar: {rows_after_non_academic}")
print(f"Filas eliminadas en subgrupo No Académico: {rows_before_non_academic - rows_after_non_academic}")


# --- Combinar los DataFrames Limpios ---
print("\nRecombinando los subgrupos limpios...")
df_fully_cleaned = pd.concat([df_academic, df_non_academic], ignore_index=True)

# --- Verificación Final ---
print("\nVerificación final de NaNs después de eliminar por subgrupo:")
final_nan_counts = df_fully_cleaned.isnull().sum()
print(final_nan_counts)


# El DataFrame 'df_fully_cleaned' es el resultado final, ahora sí completamente limpio.
print(f"\nDataFrame final 'df_fully_cleaned' tiene: {df_fully_cleaned.shape[0]} filas y {df_fully_cleaned.shape[1]} columnas.")

--- PASO 5: Eliminando filas con NaNs restantes DENTRO de subgrupos ---
Total de filas ANTES de la eliminación por subgrupo: 2441

--- Subgrupo: Académicos (Academic_or_Non_Academic == 2) ---
Filas antes de limpiar: 1904
NaNs a eliminar en columnas relevantes para Académicos:
Teaching_Hours                 37
Research_Hours                 51
Funded_Research_Activities    123
Administrative_Activities      50
dtype: int64
Filas después de limpiar: 1762
Filas eliminadas en subgrupo Académico: 142

--- Subgrupo: No Académicos (Academic_or_Non_Academic == 1) ---
Filas antes de limpiar: 537
NaNs a eliminar en columnas relevantes para No Académicos:
Job_Category                4
Highest_Education_Level    54
Career_Length_CZ           51
dtype: int64
Filas después de limpiar: 478
Filas eliminadas en subgrupo No Académico: 59

Recombinando los subgrupos limpios...

Verificación final de NaNs después de eliminar por subgrupo:
Country                            0
Version                       

#Se guarda el nuevo conjunto de datos en un excel y CSV

In [355]:
df_fully_cleaned.to_csv("labour_well_being_data.csv", index=False, encoding="utf-8")

df_fully_cleaned.to_excel("labour_well_being_data.xlsx", index=False, engine="openpyxl")
